# 09. Robustness of the country indices

Two checks, both asked for by the referees:

| | check | asked in |
|---|---|---|
| **S1** | raise the posts-per-country threshold | R1.3.a, "how dropping posts from countries with relatively low frequency would shape the results" |
| **S2** | drop every account we can identify as not a private individual | R1.3.c, "filtering for certain known institutional or government affiliated accounts"; R2.3 |

Both recompute ADI, OSI and TVR through the same `aggregate.country_indices`, so neither is
a second implementation of an index.

## The unit is a country

Every index here is computed per country: a country's own posts give its online topic
profile, which is compared against the NDC of the Party it filed under. The EU's member
states are therefore separate units sharing one NDC. This is what the figures plot and what
Table 1 regresses, and it is what makes the manuscript's counts line up — on the published
corpus, 95 countries clear the 100-post threshold and 91 of those have an NDC, which are
the two numbers the paper reports.

## Read against the claims, not the index values

A per-country correlation is the wrong statistic for S1: raising the threshold removes
countries without touching the posts behind any country that stays, so every survivor keeps
the ADI it already had and the correlation is 1.0 by construction. What the threshold can
change is **which countries are in the analysis**, and therefore any statement made across
them. So both checks are reported against the paper's actual claims:

* the abstract's leading claim, that developed-country alliances show greater communicative
  distance and less supportive discourse than developing-country ones;
* the Figure 3 claim, that G1 (emission accounting and targets) is the most under-discussed
  group online and G5 (social inclusion) the most over-discussed.

Aggregates cannot move further than the country values they average, so settling these two
also settles Figures 2, 3 and 4 without recomputing them.

**One caveat on the alliances.** Only the Umbrella and EIG memberships are written out in
the manuscript, so "developed" here means those two plus the EU's member states and "other"
is everything else. That tests the direction of the claim, not the seven-way ordering, and
has to be redone with the published lists.

## Three things that need no check

**Retweets.** The letter's summary lists "exclude RTs". The corpus contains none — the
vendor's stream carries original posts only (notebook 07, T6). One sentence in the Methods.

**The analysis window.** Restricting to calendar 2022, which the manuscript already claims,
moves nothing: ADI r = 0.9995, OSI r = 0.9843 (notebook 05, section 4).

**Posting frequency.** R1.3.c raises it, but section 4 below shows frequency does not
identify the accounts the referee is worried about in this corpus, so it is answered with
that evidence rather than with an exclusion.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [1]:
import os

import numpy as np
import pandas as pd

import config as cfg
import aggregate as ag
import regress as rg
import sensitivity as sn

O = cfg.OUTPUT_DIR
T = os.path.join(O, 'tables')
os.makedirs(T, exist_ok=True)
pd.set_option('display.width', 220)
print("output:", O)

output: …/output


## 1. Load

Posts carry the place and the account fields; recall and stance carry the measurements. The
assertions matter — a silent misalignment here would be invisible in every number below.

In [2]:
posts = pd.read_parquet(os.path.join(O, 'climate_posts_2022.parquet'),
                        columns=['tweet_id', 'country', 'user_id', 'screen_name',
                                 'source', 'statuses_count', 'user_created_at'])
recall = pd.read_parquet(os.path.join(O, 'topic_recall_2022.parquet'))
stance = pd.read_parquet(os.path.join(O, 'topic_stance_2022.parquet'))
# the post's own stance, which OSI is the plain mean of
sscore = pd.read_parquet(os.path.join(O, 'stance_2022.parquet'),
                         columns=['tweet_id', 'stance_score'])
assert (sscore.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()
sscore = sscore.stance_score
ga_chunks = pd.read_csv(os.path.join(O, 'GA_topics.csv'), low_memory=False)

assert (posts.tweet_id.to_numpy() == recall.tweet_id.to_numpy()).all()
assert (posts.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()

posts = ag.add_party(posts)
ga_party = ag.ndc_by_party(ga_chunks)
ga_country = rg.ga_by_country(ga_party)   # each country gets its Party's NDC weights

# The unit passed to country_indices is the country, not the Party.
unit = posts.shp_country

print(f"posts  {len(posts):,}   accounts {posts.user_id.nunique():,}")
print(f"NDC    {len(ga_chunks):,} chunks -> {len(ga_party)} Parties -> "
      f"{ga_country.party.nunique()} countries")
print(f"countries in the corpus: {unit.nunique()}")

posts  590,488   accounts 176,671
NDC    2,762 chunks -> 165 Parties -> 191 countries
countries in the corpus: 142


## 2. The hand labels

Notebook 08 wrote out the 200 busiest accounts; we then read each profile and assigned a
category. That table is half of S2's exclusion.

The join is on `screen_name`, not `user_id`. Twitter snowflake ids are 19 digits, and a
round trip through a spreadsheet stores them as float64, which keeps 15 to 16 significant
figures — enough to lose 81 of the 200. The check below makes that loud rather than silent:
the post counts recovered from the corpus have to equal the ones in the table, row for row.

In [3]:
lab = sn.read_labels(os.path.join(O, 'T5_top_200_accounts_labeled.csv'))
print(sn.check_labels(posts, lab))

cat = lab.groupby('category').agg(accounts=('posts', 'size'), posts=('posts', 'sum'))
cat['pct_of_top200'] = (100 * cat.posts / cat.posts.sum()).round(1)
cat['pct_of_sample'] = (100 * cat.posts / len(posts)).round(3)
print()
print(cat.sort_values('posts', ascending=False).to_string())
print(f"\ntop 200 accounts are {100 * cat.posts.sum() / len(posts):.2f}% of the sample")
print(f"non-individual among them: {cat.loc[sn.NON_INDIVIDUAL, 'posts'].sum():,} posts, "
      f"{100 * cat.loc[sn.NON_INDIVIDUAL, 'posts'].sum() / len(posts):.2f}% of the sample")
print(f"  counting every unclear account as non-individual: "
      f"{100 * cat.loc[sn.NON_INDIVIDUAL + sn.UNCLEAR, 'posts'].sum() / len(posts):.2f}%")

{'rows': 200, 'mismatched': 0, 'posts': 66304}

              accounts  posts  pct_of_top200  pct_of_sample
category                                                   
individual         122  41776           63.0          7.075
automated           33  10498           15.8          1.778
unclear             27   9100           13.7          1.541
organization        14   4230            6.4          0.716
government           4    700            1.1          0.119

top 200 accounts are 11.23% of the sample
non-individual among them: 15,428 posts, 2.61% of the sample
  counting every unclear account as non-individual: 4.15%


### Was reading the profiles by hand necessary?

`descriptives.guess_account_type` makes a first guess from the profile text, which is what
ordered the table for reading. It is a fair comparison only once the two label sets are
lined up (`sn.GUESS_TO_LABEL`), and the heuristic has no bin for "unclear", so the 27
accounts whose profile does not settle the question are unguessable by construction and are
reported separately.

In [4]:
print(sn.guess_accuracy(lab))
print()
print(pd.crosstab(lab.auto_guess.map(sn.GUESS_TO_LABEL), lab.category, margins=True).to_string())

{'n': 200, 'agreement': 0.7, 'n_determinable': 173, 'agreement_determinable': 0.8092485549132948, 'individual_vs_not': 0.74}

category      automated  government  individual  organization  unclear  All
auto_guess                                                                 
automated            31           0           2             3        0   36
government            0           1           2             0        0    3
individual            1           1         103             6       25  136
organization          1           2          15             5        2   25
All                  33           4         122            14       27  200


## 3. S2's exclusion, and why it is one exclusion rather than four

Automation can be read two ways, and each misses what the other catches:

* the **posting client** covers all 176,671 accounts, but only catches those posting through
  a scheduler such as CareerArc or dlvr.it;
* the **profile, read by hand**, is accurate but only the 200 busiest accounts were read.

Reporting them separately would invite picking the flattering one, so the exclusion is the
union: any account either method identifies as automated, institutional or governmental.

In [5]:
excl = sn.non_individual(posts, lab)
excl_up = sn.non_individual(posts, lab, include_unclear=True)

rows = []
for name, m in [('automated, by posting client', sn.by_client(posts)),
                ('automated, by hand', sn.by_label(posts, lab, ['automated'])),
                ('institutional or government, by hand',
                 sn.by_label(posts, lab, sn.INSTITUTIONAL)),
                ('S2 union of the above', excl),
                ('S2 upper bound, adding unclear', excl_up)]:
    rows.append({'exclusion': name, 'posts': int(m.sum()),
                 'pct_of_sample': round(100 * m.mean(), 2),
                 'accounts': int(posts.loc[m, 'user_id'].nunique())})
sizes = pd.DataFrame(rows).set_index('exclusion')
sizes.to_csv(os.path.join(T, 'T10_exclusion_sizes.csv'))
sizes

,posts,pct_of_sample,accounts
exclusion,,,
"automated, by posting client",34172,5.79,2461
"automated, by hand",10498,1.78,33
"institutional or government, by hand",4930,0.83,18
S2 union of the above,38968,6.60,2480
"S2 upper bound, adding unclear",48068,8.14,2507


## 4. Why posting frequency is not used as an exclusion

R1.3.c mentions "patterns across users related to posting frequency". Two frequency-based
exclusions suggest themselves — the busiest accounts in this sample, and accounts with a
high lifetime posting rate — and neither identifies what the referee is worried about.

The reason is in the last table: the hand-labelled **automated** accounts have the *lowest*
median lifetime posting rate of any category. The automation in this corpus is
single-purpose feed accounts, young and steady, while the heaviest individual users have
fifteen years of history behind them. Frequency and automation run in opposite directions
here, so the account types are identified directly instead.

In [6]:
vol, n_vol = sn.high_volume(posts, top_frac=0.01)
rate, n_rate = sn.high_rate(posts, per_day=50)
known = dict(zip(lab.screen_name, lab.category))

for name, m, n_acc in [('busiest 1% of accounts', vol, n_vol),
                       ('lifetime rate above 50 posts/day', rate, n_rate)]:
    lbl = posts.loc[m, 'screen_name'].map(known).dropna()
    share = lbl.value_counts(normalize=True) * 100
    auto_client = sn.by_client(posts)
    print(f"{name}: removes {int(m.sum()):,} posts ({100 * m.mean():.2f}%) "
          f"from {n_acc:,} accounts")
    print(f"  of the {len(lbl):,} removed posts we have labelled: "
          + ", ".join(f"{k} {v:.0f}%" for k, v in share.items()))
    print(f"  catches only {100 * (m & auto_client).sum() / auto_client.sum():.1f}% "
          f"of the posts the client check flags as automated")
    print()

print("median lifetime posting rate of the top 200 accounts, by hand label")
print(lab.groupby('category').posts_per_day_lifetime
      .agg(['median', 'max', 'size']).round(1).to_string())

busiest 1% of accounts: removes 161,445 posts (27.34%) from 1,767 accounts
  of the 66,304 removed posts we have labelled: individual 63%, automated 16%, unclear 14%, organization 6%, government 1%
  catches only 60.3% of the posts the client check flags as automated

lifetime rate above 50 posts/day: removes 28,847 posts (4.89%) from 3,937 accounts
  of the 7,556 removed posts we have labelled: individual 71%, automated 17%, unclear 12%
  catches only 3.1% of the posts the client check flags as automated

median lifetime posting rate of the top 200 accounts, by hand label
              median     max  size
category                          
automated        0.2  1342.2    33
government       3.2     5.0     4
individual      15.0   329.6   122
organization    10.2    34.3    14
unclear         11.3   776.4    27


## 5. The two checks

`min_posts` is the S1 knob and `mask` is the S2 knob, so both checks are calls to the same
function. The inclusion rule stays at more than 100 posts per country; the higher thresholds
are reported, not adopted.

In [7]:
checks = [
    ('S2 no non-individual accounts', ~excl, None),
    ('S2 upper bound, adding unclear', ~excl_up, None),
    ('S1 more than 200 posts', None, 200),
    ('S1 more than 500 posts', None, 500),
    ('S1 more than 1,000 posts', None, 1000),
    ('S1 more than 2,000 posts', None, 2000),
]
summary, idx_by_check = sn.run(posts.assign(party=unit), recall, stance, ga_country,
                               checks, stance_score=sscore)
base = idx_by_check['all posts']
print(f"baseline: {len(base)} countries")
summary.round(4)

baseline: 100 countries


,posts,pct_of_posts,parties,ADI_mean,OSI_mean,ADI_pearson,ADI_spearman,OSI_pearson,shared_parties
check,,,,,,,,,
all posts,584730,99.0249,100,0.1000,0.7250,1.0000,1.0000,1.0000,100
S2 no non-individual accounts,545911,92.4508,100,0.0998,0.7231,0.9998,0.9994,0.9974,100
"S2 upper bound, adding unclear",536966,90.9360,100,0.0999,0.7232,0.9997,0.9992,0.9961,100
S1 more than 200 posts,582118,98.5825,82,0.1033,0.7317,1.0000,1.0000,1.0000,82
S1 more than 500 posts,572275,96.9156,54,0.1058,0.7484,1.0000,1.0000,1.0000,54
"S1 more than 1,000 posts",561098,95.0228,41,0.1050,0.7488,1.0000,1.0000,1.0000,41
"S1 more than 2,000 posts",537865,91.0882,24,0.1101,0.7478,1.0000,1.0000,1.0000,24


## 6. Do the claims survive?

The columns that matter. `ADI_gap` is developed minus other and the abstract says it is
positive; `OSI_gap` is other minus developed and the abstract says it is positive too;
`tvr_order` runs from the most under-discussed group online to the most over-discussed, and
the text says G1 comes first and G5 near the end.

In [8]:
claims = pd.DataFrame({k: sn.conclusions(v) for k, v in idx_by_check.items()}).T
claims.index.name = 'check'
claims.to_csv(os.path.join(T, 'T11_claims_under_each_check.csv'))

cols = ['parties', 'n_developed', 'ADI_developed', 'ADI_other', 'ADI_gap',
        'OSI_developed', 'OSI_other', 'OSI_gap']
print(claims[cols].astype(float).round(5).to_string())
print()
print(claims[['tvr_order', 'G1_most_under', 'G5_G6_most_over']].to_string())
print()
bad = claims[(claims.ADI_gap <= 0) | (claims.OSI_gap <= 0) | (~claims.G1_most_under.astype(bool))]
print("checks where a claim fails:", "none" if bad.empty else list(bad.index))

                                parties  n_developed  ADI_developed  ADI_other  ADI_gap  OSI_developed  OSI_other  OSI_gap
check                                                                                                                     
all posts                         100.0         33.0        0.10689    0.09656  0.01033        0.69583    0.73942  0.04359
S2 no non-individual accounts     100.0         33.0        0.10673    0.09640  0.01032        0.69020    0.73928  0.04908
S2 upper bound, adding unclear    100.0         33.0        0.10674    0.09650  0.01024        0.69162    0.73876  0.04715
S1 more than 200 posts             82.0         29.0        0.11015    0.09948  0.01067        0.69177    0.75352  0.06176
S1 more than 500 posts             54.0         22.0        0.11172    0.10178  0.00994        0.70732    0.77669  0.06937
S1 more than 1,000 posts           41.0         16.0        0.11210    0.10042  0.01168        0.69666    0.78222  0.08556
S1 more than 2,0

In [9]:
### Group-level TVR under each check, to see how close to zero the middle groups sit

g = claims[[f'G{i}_TVR' for i in range(1, 7)]].astype(float)
print(g.round(3).to_string())
print()
print("sign, per check")
print(np.sign(g).astype(int).to_string())

                                G1_TVR  G2_TVR  G3_TVR  G4_TVR  G5_TVR  G6_TVR
check                                                                         
all posts                       -0.661   0.197   0.286  -0.159   1.074   0.918
S2 no non-individual accounts   -0.659   0.198   0.288  -0.160   1.071   0.916
S2 upper bound, adding unclear  -0.659   0.198   0.287  -0.159   1.073   0.915
S1 more than 200 posts          -0.683   0.220   0.324  -0.174   1.070   0.929
S1 more than 500 posts          -0.667   0.185   0.450  -0.188   1.073   0.861
S1 more than 1,000 posts        -0.671   0.217   0.397  -0.241   1.072   0.895
S1 more than 2,000 posts        -0.621   0.193   0.610  -0.235   1.001   0.770

sign, per check
                                G1_TVR  G2_TVR  G3_TVR  G4_TVR  G5_TVR  G6_TVR
check                                                                         
all posts                           -1       1       1      -1       1       1
S2 no non-individual accounts      

## 7. How far can anything move?

For S2 the per-country comparison is meaningful, so it is worth knowing how far individual
countries travel. Reported against each index's own spread across countries, because an absolute
change means nothing without it.

S1 is left out of this table: it changes no country's index by construction.

In [10]:
rows = []
for label in ['S2 no non-individual accounts', 'S2 upper bound, adding unclear']:
    idx = idx_by_check[label]
    shared = base.index.intersection(idx.index)
    r = {'check': label, 'parties': len(shared)}
    for col in ['ADI', 'OSI']:
        d = (idx.loc[shared, col] - base.loc[shared, col]).abs()
        sd = base.loc[shared, col].std()
        rk = idx.loc[shared, col].rank(ascending=False) - base.loc[shared, col].rank(ascending=False)
        r[f'{col}_pearson'] = base.loc[shared, col].corr(idx.loc[shared, col])
        r[f'{col}_spearman'] = base.loc[shared, col].corr(idx.loc[shared, col], method='spearman')
        r[f'{col}_mean_change_pct_of_sd'] = 100 * d.mean() / sd
        r[f'{col}_max_change'] = d.max()
        r[f'{col}_max_rank_shift'] = rk.abs().max()
    rows.append(r)
move = pd.DataFrame(rows).set_index('check')
move.to_csv(os.path.join(T, 'T12_index_movement.csv'))
move.round(4).T

check,S2 no non-individual accounts,"S2 upper bound, adding unclear"
parties,100.0000,100.0000
ADI_pearson,0.9998,0.9997
ADI_spearman,0.9994,0.9992
ADI_mean_change_pct_of_sd,0.7973,0.9908
ADI_max_change,0.0064,0.0074
ADI_max_rank_shift,4.0000,6.0000
OSI_pearson,0.9974,0.9961
OSI_spearman,0.9964,0.9957
OSI_mean_change_pct_of_sd,2.8452,3.8735
OSI_max_change,0.0691,0.0691


In [11]:
### Which countries move most under S2

d = pd.DataFrame({'ADI_base': base.ADI,
                  'ADI_excl': idx_by_check['S2 no non-individual accounts'].ADI,
                  'n_posts': base.n_posts}).dropna()
d['change'] = d.ADI_excl - d.ADI_base
print(d.reindex(d.change.abs().sort_values(ascending=False).index).head(8).round(4).to_string())

                          ADI_base  ADI_excl  n_posts  change
party                                                        
United States of America    0.1595    0.1531   225138 -0.0064
Ecuador                     0.0592    0.0569      186 -0.0024
Zambia                      0.1895    0.1877     1050 -0.0019
Israel                      0.0773    0.0757      466 -0.0016
Malaysia                    0.2056    0.2072     1847  0.0016
France                      0.0812    0.0826     4313  0.0014
Brazil                      0.1395    0.1383     1208 -0.0012
Argentina                   0.1321    0.1309      347 -0.0012


## 8. Which countries the threshold removes

The threshold trades countries for almost no posts, and what it removes is not a random
sample of them: at more than 500 posts, nearly every country dropped is outside the developed
alliances. By the referee's reasoning that should be where the contrast is most at risk;
section 6 shows the gap widens instead, which says those low-count countries were diluting
the contrast rather than creating it.

In [12]:
for t in [200, 500, 1000, 2000]:
    dropped = base.index[base.n_posts <= t]
    n_dev = int(pd.Index(dropped).isin(sn.DEVELOPED).sum())
    print(f"more than {t:>5,} posts: {len(base) - len(dropped):>3} countries kept, "
          f"{len(dropped):>3} dropped ({n_dev} of them in a developed alliance)")
print()
print("dropped at more than 500 posts:")
print(" ", ", ".join(base.index[base.n_posts <= 500]))

more than   200 posts:  82 countries kept,  18 dropped (4 of them in a developed alliance)
more than   500 posts:  54 countries kept,  46 dropped (11 of them in a developed alliance)
more than 1,000 posts:  41 countries kept,  59 dropped (17 of them in a developed alliance)
more than 2,000 posts:  24 countries kept,  76 dropped (21 of them in a developed alliance)

dropped at more than 500 posts:
  Albania, Algeria, Argentina, Azerbaijan, Bahamas, Bahrain, Barbados, Botswana, Cameroon, Chile, Costa Rica, Croatia, Cyprus, Czech Republic, Democratic Republic of the Congo, Dominican Republic, Ecuador, Gambia, Georgia, Guatemala, Hungary, Iceland, Iraq, Israel, Jordan, Kazakhstan, Kuwait, Lebanon, Lesotho, Liberia, Maldives, Malta, Mozambique, Namibia, Oman, Republic of Korea, Romania, Russian Federation, Serbia, Sierra Leone, Slovakia, South Sudan, Trinidad and Tobago, Tunisia, Ukraine, Vietnam


## 9. Output

In [13]:
summary.to_csv(os.path.join(T, 'T13_sensitivity_summary.csv'))

SLUG = {'S2 no non-individual accounts': 's2',
        'S2 upper bound, adding unclear': 's2_upper',
        'S1 more than 200 posts': 's1_gt200',
        'S1 more than 500 posts': 's1_gt500',
        'S1 more than 1,000 posts': 's1_gt1000',
        'S1 more than 2,000 posts': 's1_gt2000'}
for label, idx in idx_by_check.items():
    if label == 'all posts':
        continue
    idx.to_csv(os.path.join(O, f'indices_2022_by_country_{SLUG[label]}.csv'))

print("wrote")
for f in ['T10_exclusion_sizes.csv', 'T11_claims_under_each_check.csv',
          'T12_index_movement.csv', 'T13_sensitivity_summary.csv']:
    print("  tables/" + f)
for label in SLUG:
    print(f"  indices_2022_by_country_{SLUG[label]}.csv")
summary.round(4)

wrote
  tables/T10_exclusion_sizes.csv
  tables/T11_claims_under_each_check.csv
  tables/T12_index_movement.csv
  tables/T13_sensitivity_summary.csv
  indices_2022_by_country_s2.csv
  indices_2022_by_country_s2_upper.csv
  indices_2022_by_country_s1_gt200.csv
  indices_2022_by_country_s1_gt500.csv
  indices_2022_by_country_s1_gt1000.csv
  indices_2022_by_country_s1_gt2000.csv


,posts,pct_of_posts,parties,ADI_mean,OSI_mean,ADI_pearson,ADI_spearman,OSI_pearson,shared_parties
check,,,,,,,,,
all posts,584730,99.0249,100,0.1000,0.7250,1.0000,1.0000,1.0000,100
S2 no non-individual accounts,545911,92.4508,100,0.0998,0.7231,0.9998,0.9994,0.9974,100
"S2 upper bound, adding unclear",536966,90.9360,100,0.0999,0.7232,0.9997,0.9992,0.9961,100
S1 more than 200 posts,582118,98.5825,82,0.1033,0.7317,1.0000,1.0000,1.0000,82
S1 more than 500 posts,572275,96.9156,54,0.1058,0.7484,1.0000,1.0000,1.0000,54
"S1 more than 1,000 posts",561098,95.0228,41,0.1050,0.7488,1.0000,1.0000,1.0000,41
"S1 more than 2,000 posts",537865,91.0882,24,0.1101,0.7478,1.0000,1.0000,1.0000,24


## Scope

Done: S1 and S2, read against the abstract's leading claim and the Figure 3 claim.

Not done, by decision: recomputing ADI over the 38 topics instead of the six groups,
variants of the OSI definition, alternative divergence functions, splitting the EU into its
27 members, a label-shuffling placebo, restricting to the strong filter rules, equalising
the number of posts per Party, and varying the number of topics or the topic-model seed. The
last would require refitting BERTopic and rerunning BERTScore over the whole corpus; the
topic model is a single fit with the number of topics chosen by coherence, which belongs in
the limitations.

Still open: the seven-alliance memberships, so section 6 can use the paper's own grouping
rather than the Umbrella + EIG + EU proxy.

The regression in notebook 06 is the main analysis, not a robustness check. R1.3.a asks for
"tweet count as control **or** countries above a higher threshold", and S1 answers it.